# Retention times for the C27 tetrahydroxy library

One-off companion to `ISF_plot.ipynb` (step 3): the loop in `ISF_plot.ipynb` left most C27 tetrahydroxy scans without a retention time. This notebook rebuilds `C27_tetrahydroxy_table_for_artifacts_estimation.tsv` by matching on original path + scan, before the ion-form screen is run.

In [ ]:
import os

# Root of the local project folder (the one holding Falcon/, Queries_final_mgf/,
# final_mgfs_library_generation/, ...). Edit this single line to run elsewhere.
PROJECT_DIR = '/media/dorrestein/Helena2/Bile_acids_update'

In [ ]:
import pandas as pd
import polars as pl

In [ ]:
#first, let's import the C23 library to know which are the 'identifier' values that made it to the final library

#import
df = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C27_annotation_combined_scan.tsv', sep='\t')
#harmonize the BA_core column
df['BA_core'] = df['identifier'].str.split(':').str[2].replace('_input_falcon_charge', '', regex=True)
df['BA_core'] = df['BA_core'].replace('Mono', 'mono', regex=True)
df['BA_core'] = df['BA_core'].replace('Non', 'non', regex=True)
df['BA_core'] = df['BA_core'].replace('Tri', 'tri', regex=True)
df['BA_core'] = df['BA_core'].replace('Aceto', 'C26', regex=True)
df['BA_core'] = df['BA_core'].replace('C28_alcohol_3OH_input_falcon_383_C28O3_charge', 'C28_alcohol_3OH', regex=True)
df['BA_core'] = df['BA_core'].replace('C28_alcohol_2OH_input_falcon_367_C28O2_charge', 'C28_alcohol_2OH', regex=True)
df['BA_core'] = df['BA_core'].replace('C28_alcohol_1OH_input_falcon_351_C28O1_charge', 'C28_alcohol_1OH', regex=True)
df['BA_core'] = df['BA_core'].replace('alkylamine', 'alkamine', regex=True)

df = df[df['BA_core'].str.contains('C27_tetra')]

In [ ]:
#filter polars dataframe and convert to pandas
#import massql results combined table
combined = pl.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/combined_all_massql_results_cleaned.tsv', separator='\t')
# combined = combined.filter(pl.col('BA_core').str.starts_with('C27_tetra'))
combined = combined.to_pandas()

In [ ]:
#import pre and post falcon tables
post_falcon = pd.read_csv(f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_tetrahydroxy/falcon_output/C27_tetrahydroxy_falcon_eps01_minsamples2.csv', sep=',', skiprows=25)
post_falcon['scan_combined_mgf'] = post_falcon['identifier'].str.split(':').str[-1]

pre_falcon = pd.read_csv(f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_tetrahydroxy/download_mgfs/final_mgf/C27_tetrahydroxy_input_falcon_charge_annotation_table.tsv', sep='\t')

# merge
post_falcon['scan_combined_mgf'] = post_falcon['scan_combined_mgf'].astype(str)
pre_falcon['scan_combined_mgf'] = pre_falcon['scan_combined_mgf'].astype(str)

merged = pd.merge(pre_falcon, post_falcon, on='scan_combined_mgf', how='left')

#now let's merge with df to know which are the cluster indexes that made it to the final library
clusters_to_keep = pd.merge(df['identifier'], merged[['identifier', 'cluster']], on='identifier', how='left')
clusters_to_keep = clusters_to_keep['cluster'].to_list()

#let's keep only these clusters in the merged table
merged = merged[merged['cluster'].isin(clusters_to_keep)]

#and finally merge with the combined dataframe to have the retention time information
merged['original_path_scan'] = merged['original_path'] + '_' + merged['original_scan'].astype(str)
combined['original_path_scan'] = combined['original_path'] + '_' + combined['scan'].astype(str)

final_merged = pd.merge(merged, combined[['original_path_scan', 'rt']], on='original_path_scan', how='left')

final_merged.to_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/BA_core_merged_files_for_plot/C27_tetrahydroxy_table_for_artifacts_estimation.tsv', sep='\t', index=False)